<a href="https://colab.research.google.com/github/mrbedoya/ml-2026/blob/main/03_RegresionLineal(2da_Parte).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regresión Lineal (2da parte)

> **Regresión lineal múltiple** para predecir el precio de venta de vehículos usados. El proceso sigue estos pasos:
>
> 1. **Todas las variables numéricas** como punto de partida (incluidas `Year` y `Kilometer`). Las correlaciones con el precio se usan para explorar los datos, no para descartar variables.
> 2. **Logaritmo del precio** como variable de salida, con las variables de entrada en su escala original.
> 3. **VIF con intercepto** (`check_vif(..., intercepto=True)`) para detectar la colinealidad.
> 4. **Selección hacia atrás**, una variable a la vez: primero por VIF (> 10) y luego por *p-value* (> 0.05) con errores estándar robustos (HC3).
> 5. **Diagnóstico de los supuestos** del modelo (homocedasticidad, normalidad de los residuos, observaciones influyentes).
> 6. **Evaluación en dólares** con el conjunto de prueba real (`01_carDetailsV4_Prue.csv`).
>
> Usa los datos preparados en `01_PreparaciónDataset01Regresion` (categorías raras de `Owner` y `Seller Type` agrupadas en `Otro`).

## 1. Librerias y configuraciones previas

In [ ]:
# Tratamiento de datos
# ==============================================================================
import pandas as pd
import numpy as np


# Preprocesado y modelado
# ==============================================================================

#Separar los datos entrenamiento y prueba
from sklearn.model_selection import train_test_split

#Escalar Variables
from sklearn.preprocessing import MinMaxScaler

#Creación de modelo
from statsmodels import api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera

#Evaluación del modelo
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                            median_absolute_error)

# Gráficos
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# Interactuar con el sistema operativo y solicitudes HTTP
# ==============================================================================
import os
import sys
import requests
from urllib.parse import urljoin

# Configuración warnings
# ==============================================================================
import warnings
# Se ocultan solo los avisos informativos de versiones; los demás avisos siguen visibles
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
# Aviso de matplotlib que genera internamente sm.qqplot al personalizar el color de los puntos (no afecta el resultado)
warnings.filterwarnings('ignore', message='.*redundantly defined by the')

In [ ]:
# Parametros globales
# ==============================================================================

# URL base de GitHub para el repositorio
GITHUB_BASE_URL = 'https://raw.githubusercontent.com/mrbedoya/ml-2026/main/'

# Nombres de los archivos requeridos
DATASET_FILE_NAME = '01_carDetailsV4_Prep.csv'
DATASET_FILE_NAME_PRUE = '01_carDetailsV4_Prue.csv'
UTILS_FILE_NAME = 'funciones.py'

# Nombres de los directorios para utilidades y datasets
DATASETS_PATH = 'datasets/'
UTILS_PATH = 'utils/'

# URL completa del dataset y utils en GitHub (urljoin evita errores con las barras)
github_dataset_url = urljoin(GITHUB_BASE_URL, DATASETS_PATH + DATASET_FILE_NAME)
github_dataset_url_prue = urljoin(GITHUB_BASE_URL, DATASETS_PATH + DATASET_FILE_NAME_PRUE)
github_utils_url = urljoin(GITHUB_BASE_URL, UTILS_PATH + UTILS_FILE_NAME)

# Rutas locales completas de datasets y utils
local_dataset_path = os.path.join(DATASETS_PATH, DATASET_FILE_NAME)
local_dataset_path_prue = os.path.join(DATASETS_PATH, DATASET_FILE_NAME_PRUE)
local_utils_path = os.path.join(UTILS_PATH, UTILS_FILE_NAME)

### (Opcional) Cargar datos desde Google Drive

La celda siguiente muestra una **alternativa** para trabajar con archivos guardados en Google Drive, en vez de descargarlos desde GitHub. Se deja comentada porque en este notebook usamos la descarga directa desde el repositorio (más fácil de reproducir para cualquier estudiante), pero queda como referencia por si prefieres montar tu propio Drive.

In [ ]:
## Cargar datos desde Google Drive
## ==============================================================================

#from google.colab import drive

## Montar Google Drive
#drive.mount('/gdrive')

## Establece el directorio de trabajo dentro de directorio especificado en Google Drive
#os.chdir("/gdrive/MyDrive/01_Academia/02_Cursos/20260401_ApredinzajeAutomáticoI_UdeA/")

## Listar archivos de directorio
##!ls

In [ ]:
# Funcion local para la carga de archivos
# ==============================================================================

def download_file(url, path, description, force=False, timeout=30):
    '''
    Descarga un archivo desde una URL y lo guarda localmente en un path.
    Si el archivo ya existe no se vuelve a descargar (usar force=True para obligarlo).

    Args:
        url (str): URL del archivo
        path (str): Ruta local de destino
        description (str): Descripción del archivo
        force (bool): Si es True, descarga aunque el archivo ya exista
        timeout (int): Segundos máximos de espera de la descarga
    Returns:
        None
    '''
    if os.path.exists(path) and not force:
        print(f"{description} ya existe en: {path} (no se vuelve a descargar)")
        return

    print(f"Descargando {description} desde: {url}")

    # Realiza una solicitud HTTP GET a la URL proporcionada
    response = requests.get(url, timeout=timeout)

    # Lanza una excepción si la descarga falla (código de estado HTTP no 200)
    response.raise_for_status()

    # Abre el archivo local en modo de escritura binaria ('wb')
    with open(path, 'wb') as f:
        # Escribe el contenido binario de la respuesta HTTP en el archivo local
        f.write(response.content)

    # Imprime un mensaje confirmando que la descarga fue exitosa y la ruta de guardado
    print(f"{description} descargado exitosamente a: {path}")

In [ ]:
# Configuración entorno Colab
# ==============================================================================

# Creación de directorios locales
os.makedirs(DATASETS_PATH, exist_ok=True)
os.makedirs(UTILS_PATH, exist_ok=True)

# Descarga de archivos
download_file(github_dataset_url, local_dataset_path, "dataset")
download_file(github_dataset_url_prue, local_dataset_path_prue, "dataset de prueba")
download_file(github_utils_url, local_utils_path, "funciones")

## 2. Funciones

En esta sección importamos las funciones reutilizables de `funciones.py` (descargado arriba) y definimos las funciones propias de este notebook: construir el modelo y calcular las métricas en dólares.

In [ ]:
# Funciones externas
# ==============================================================================

# Añade la ruta definida en 'UTILS_PATH' al 'sys.path' para que Python pueda encontrar módulos
sys.path.append(UTILS_PATH)

# Importa funciones externas
from funciones import multiple_plot, tidy_corr_matrix, check_vif

In [ ]:
# Función para construir un modelo de regresión lineal por medio de statsmodels.api

def build_model(X, y, robusto=True):
    '''
    Crea y entrena un modelo de regresión lineal (OLS) con statsmodels e imprime su resumen.

    Args:
        X (DataFrame): variables de entrada (a las que se les adiciona la constante)
        y (Series): variable de salida
        robusto (bool): si es True usa errores estándar robustos HC3. Los coeficientes y el R2 son los
                        mismos; cambian los errores estándar, los p-values y los intervalos de confianza,
                        que siguen siendo válidos aunque haya heterocedasticidad.
    Returns:
        tuple: (modelo ajustado, 'X' con la constante adicionada)
    '''

    #A los datos de entrada 'X' se le adiciona una constante
    X = sm.add_constant(X)

    #Se crea y se entrena el modelo de regresión lineal (statsmodels.api)
    modelo = sm.OLS(y, X).fit(cov_type='HC3' if robusto else 'nonrobust')

    #imprime el resumen del modelo creado
    print(modelo.summary())

    return modelo, X

In [ ]:
# Función para calcular las métricas de error en la escala original del precio (dólares)

def metricas_precio(precio_real, precio_pred):
    '''
    Calcula R2, RMSE, MAE y MedAE entre el precio real y el precio predicho (ambos en dólares).
    '''
    return {'R2': r2_score(precio_real, precio_pred),
            'RMSE': np.sqrt(mean_squared_error(precio_real, precio_pred)),
            'MAE': mean_absolute_error(precio_real, precio_pred),
            'MedAE': median_absolute_error(precio_real, precio_pred)}

## 3. Carga del dataset

In [ ]:
#Se crea un dataframe d con los datos obtenidos de archivo de entrada
d = pd.read_csv(local_dataset_path)

In [ ]:
d.info()

## 4. Visualización de datos

### Variables de entrada

In [ ]:
#Lista de variables categóricas
catCols = d.select_dtypes(include = ["object", 'category']).columns.tolist()

d[catCols].head(2)

In [ ]:
#Lista de variables numéricas

numCols=d.select_dtypes(include = ['float64','float64','int32','int64']).columns.tolist()
d[numCols].head(2)

In [ ]:
##Visualización de frecuencia de instancias para variables categóricas
multiple_plot(3, d , catCols, None, 'countplot', 'Frecuencia de instancias para variables categóricas',30)

In [ ]:
#Visualización de variables numéricas con respecto al precio
multiple_plot(1, d , numCols, None, 'scatterplot', 'Relación entre las variables numéricas',30)

### Variable de salida

In [ ]:
#Histograma y caja de bigotes del precio de los vehículos

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.title('Distribución de precio de vehículos')
sns.histplot(d.Price, kde=True, stat='density')

plt.subplot(1,2,2)
plt.title('Diferencia de precios de vehículos')
sns.boxplot(x=d.Price, palette='nipy_spectral')

plt.show()

In [ ]:
pd.set_option('display.float_format', lambda x: '%.0f' % x)

print(d.Price.describe(percentiles = [0.25,0.50,0.75,0.85,0.90,0.95,0.975,1]))

In [ ]:
# Eliminar el valor 'Price' de la lista
numCols.remove('Price')

## 5. Preparación de datos

### Transformación de la variable de salida

El precio tiene una distribución sesgada a la derecha y la variación de los errores crece con el precio. Con el **logaritmo** del precio la distribución es más simétrica, la varianza es más estable y los coeficientes se interpretan como cambios porcentuales en el precio. Con este conjunto de datos el logaritmo ajusta mejor que otras transformaciones como la raíz cúbica o la raíz cuadrada.

In [ ]:
#Transformación y = ln(y) Logaritmo natural

d['Price'] = np.log(d['Price'])

In [ ]:
#Histograma y caja de bigotes del logaritmo del precio de los vehículos

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.title('Distribución del logaritmo del precio')
sns.histplot(d.Price, kde=True, stat='density')

plt.subplot(1,2,2)
plt.title('Caja de bigotes del logaritmo del precio')
sns.boxplot(x=d.Price, palette='nipy_spectral')

plt.show()

### Transformaciones a las variables de entrada

Las variables de entrada se dejan en su escala original: elevarlas al cuadrado (`Year`, `Length`, `Width`, `Fuel Tank Capacity`) no mejora el ajuste de forma apreciable y hace más difícil interpretar los coeficientes. Si la gráfica de residuos vs. predicción mostrara curvatura, se puede volver a evaluar una transformación.

### Correlaciones (exploración)

Las correlaciones con el precio sirven para **explorar** los datos, no para descartar variables: una variable con una correlación moderada (por ejemplo `Year`) puede ser muy importante al combinarla con las demás. Por eso todas las variables numéricas pasan a la selección del modelo.

In [ ]:
#Generación de la matriz de correlación (incluye la variable de salida 'Price')
matrizCorr = d[['Price']+numCols].corr()

In [ ]:
# Mapa de calor (Heatmap) de la matriz de correlaciones

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 6))

sns.heatmap(
    matrizCorr,
    annot     = True,
    cbar      = False,
    annot_kws = {"size": 8},
    vmin      = -1,
    vmax      = 1,
    center    = 0,
    cmap      = 'RdBu', #sns.diverging_palette(20, 220, n=200),
    square    = True,
    ax        = ax
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation = 45,
    horizontalalignment = 'right',
)


ax.tick_params(labelsize = 8)

In [ ]:
#Matriz de correlación ordenada

pd.set_option('display.float_format', lambda x: '%.4f' % x)

#Creación de la matriz
df_corr = tidy_corr_matrix(matrizCorr)

#Correlaciones de cada variable numérica con el precio
df_filtrado = df_corr.loc[df_corr['variable_2'] == 'Price']
df_filtrado[['variable_1', 'r', 'abs_r']]

### Creación de Variables Dummies

Con `drop_first=True` se elimina una categoría de referencia por variable (la primera alfabéticamente: `First`, `Individual`, `Diesel`, `Automatic`, `AWD`), que en estos datos es siempre una categoría con muchas filas.

In [ ]:
# Aplicación de la función de get_dummies

d = pd.get_dummies(d, dtype=int, drop_first=True)

In [ ]:
d.info()

## 6. Creación del modelo

### Dividir el conjunto de datos

In [ ]:
#Se establece las variables de entrada 'X' y la variable de salida 'y'

X = d.drop(columns='Price')
y = d['Price']

In [ ]:
# División de los datos en train y test

#En este caso, el 80% de los datos se utilizarán para entrenar el modelo.

X_train, X_test, y_train, y_test = train_test_split(
                                        X,
                                        y,
                                        train_size   = 0.8,
                                        random_state = 1234,
                                        shuffle      = True
                                    )

### Escalar Variables

La regresión lineal por mínimos cuadrados **no necesita** escalar las variables: el escalado no cambia el R², los p-values ni el VIF (solo cambia la escala de los coeficientes). Se escala para mantener la misma preparación que en los demás notebooks del curso.

In [ ]:
#Se establecen las variables numéricas a escalar

#Num_vars contiene las variables numéricas de entrada (se conserva el orden de numCols)
num_vars = [col for col in numCols if col in X.columns]

print(num_vars)

In [ ]:
#Escalar Variables númericas

pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Se crea un objeto MinMaxScaler
scaler = MinMaxScaler()

# Se escalan los valores del dataset entrenamiento y prueba de las columnas numéricas
X_train[num_vars] = scaler.fit_transform(X_train[num_vars])
X_test[num_vars] = scaler.transform(X_test[num_vars])

X_train[num_vars].head()

### Selección de variables

#### Colinealidad

El VIF (*Variance Inflation Factor*) mide cuánto se explica una variable a partir de las demás. Se calcula **con intercepto** (`intercepto=True`): sin intercepto el VIF de variables escaladas entre 0 y 1 sale inflado y puede llevar a eliminar variables que no están colineales. Si existe alguna *feature* con VIF > 10 (siendo estrictos, VIF > 5) se convierte en candidata a eliminar; un VIF `inf` indica colinealidad perfecta.

In [ ]:
#Se aplica la funcion de Factor de Inflación de la Varianza (VIF- Variance Inflation Factor) para identificar colinealidad
#entre las variables (con intercepto)

check_vif(X_train, intercepto=True)

Con este conjunto de datos ninguna variable supera el umbral de 10, por lo que **no se elimina ninguna por colinealidad**. Los VIF más altos corresponden a las variables que miden el tamaño del vehículo (`Engine`, `Length`, `Width`, `Fuel Tank Capacity`): están correlacionadas entre sí, pero aportan información relevante y su VIF es aceptable.

Si en alguna ejecución una variable superara 10, se elimina **solo la de mayor VIF** (`X_train = X_train.drop(columns=['variable'])`), se vuelve a calcular el VIF y se repite hasta que ninguna lo supere. Un VIF `inf` indica colinealidad perfecta: esas variables se resuelven primero.

#### Significancia estadística (p-value)

Con la colinealidad controlada se revisan los *p-values*. Se ajusta el modelo, se identifica la variable con **mayor** *p-value* y, si es mayor a 0.05 (no hay suficiente evidencia de que tenga un efecto significativo sobre el precio una vez consideradas las demás), se elimina. Se elimina **una sola variable por paso** y se vuelve a ajustar el modelo, porque al quitarla cambian los *p-values* de las demás.

Los *p-values* se calculan con errores estándar robustos (HC3), que siguen siendo válidos aunque la varianza de los errores no sea constante. Se revisan después de la colinealidad porque esta infla los errores estándar y hace poco confiables los *p-values*.

##### Paso 1: modelo con todas las variables

In [ ]:
# Paso 1: modelo con todas las variables
X_train_modelo = X_train.copy()
modelo_paso, _ = build_model(X_train_modelo, y_train)

# Variables con mayor p-value (candidatas a eliminar)
modelo_paso.pvalues.drop('const').sort_values(ascending=False).head(3)

La variable con mayor *p-value* es `Fuel Type_Otro` y es mayor a 0.05: una vez consideradas las demás variables, no hay evidencia de que pertenecer a la categoría `Otro` de combustible modifique el precio. Se elimina solo esta variable y se vuelve a ajustar el modelo.

In [ ]:
#Se elimina la variable 'Fuel Type_Otro'
X_train_modelo = X_train_modelo.drop(columns=['Fuel Type_Otro'])

##### Paso 2: modelo sin `Fuel Type_Otro`

In [ ]:
# Paso 2: modelo sin `Fuel Type_Otro`
modelo_paso, _ = build_model(X_train_modelo, y_train)

# Variables con mayor p-value (candidatas a eliminar)
modelo_paso.pvalues.drop('const').sort_values(ascending=False).head(3)

Al volver a ajustar, la variable con mayor *p-value* es `Height` y es mayor a 0.05: la altura no aporta información adicional sobre el precio una vez consideradas el resto de las medidas del vehículo. Se elimina.

In [ ]:
#Se elimina la variable 'Height'
X_train_modelo = X_train_modelo.drop(columns=['Height'])

##### Paso 3: modelo sin `Fuel Type_Otro` y `Height`

In [ ]:
# Paso 3: modelo sin `Fuel Type_Otro` y `Height`
modelo_paso, _ = build_model(X_train_modelo, y_train)

# Variables con mayor p-value (candidatas a eliminar)
modelo_paso.pvalues.drop('const').sort_values(ascending=False).head(3)

La variable con mayor *p-value* es `Seller Type_Otro` y es mayor a 0.05 (no hay evidencia suficiente de que el tipo de vendedor `Otro` cambie el precio). Se elimina. Después de este paso todas las variables restantes tienen *p-value* menor a 0.05, por lo que la selección termina.

In [ ]:
#Se elimina la variable 'Seller Type_Otro'
X_train_modelo = X_train_modelo.drop(columns=['Seller Type_Otro'])

### Creación del modelo

Se entrena el modelo final con las variables que quedaron después de la selección: en la tabla de coeficientes todas tienen *p-value* menor a 0.05. Con scikit-learn (`LinearRegression`) se obtienen los mismos coeficientes, pero no el resumen estadístico (*p-values*, intervalos de confianza), por eso se usa statsmodels.

In [ ]:
# Variables finales del modelo
columnas_finales = list(X_train_modelo.columns)
print(f"Variables iniciales: {X_train.shape[1]} | variables finales: {len(columnas_finales)}")
print(columnas_finales)

#Llamamos la funcion build_model definida al principio de este notebook que crea y entrena el modelo
#con las variables finales (errores estándar robustos HC3)

modelLRegresion, X_train_const = build_model(X_train_modelo, y_train)

### Diagnóstico de los supuestos del modelo

La regresión lineal da resultados confiables cuando los **errores del modelo** (los *residuos*: valor predicho menos valor real, en escala logarítmica) se comportan de cierta manera. Las siguientes pruebas revisan si eso ocurre. Cada prueba parte de una afirmación (la *hipótesis nula*) y calcula un *p-value*: **si el p-value es menor a 0.05 se "rechaza" la afirmación**, es decir, hay evidencia de que no se cumple.

#### 1. Homocedasticidad (Breusch-Pagan)

- **Afirmación que prueba:** la variabilidad de los errores es la misma para todos los autos (el modelo se equivoca "más o menos igual" en autos baratos y caros).
- **Si se rechaza:** hay *heterocedasticidad*: los errores son más grandes en unos autos que en otros (por ejemplo, en los más caros). Las predicciones y los coeficientes siguen siendo válidos, pero los errores estándar y los *p-values* "clásicos" dejan de ser confiables.
- **Qué se hace:** usar errores estándar robustos (HC3), como en este notebook.

#### 2. Normalidad de los residuos (Jarque-Bera)

- **Afirmación que prueba:** los errores tienen forma de campana: simétricos alrededor de 0 y con muy pocos errores grandes.
- **Si se rechaza:** hay más errores extremos de lo normal, o los errores se inclinan hacia un lado. Con muchas observaciones (cientos o más) los coeficientes y los *p-values* siguen siendo aproximadamente válidos; lo que se ve afectado es sobre todo la precisión de los intervalos de predicción de un auto individual.
- **Cómo interpretarlo:** con muchos datos la prueba rechaza incluso por desviaciones pequeñas. Conviene mirar también la gráfica Q-Q de más abajo: si los puntos siguen la línea, la desviación es leve.

#### 3. Independencia de los errores (Durbin-Watson)

- **Qué mide:** si el error de una observación se parece al de la observación vecina. El estadístico va de 0 a 4 y equivale aproximadamente a `2 × (1 − correlación entre errores consecutivos)`:
  - **Cerca de 2** (entre 1.5 y 2.5): la correlación entre errores consecutivos es casi 0, es decir, los errores son independientes. Es lo que se espera. Es 2 porque `2 × (1 − 0) = 2`.
  - **Menor a 1.5:** el error de una observación se parece al de la anterior (autocorrelación positiva).
  - **Mayor a 2.5:** el error tiende a cambiar de signo de una observación a la siguiente (autocorrelación negativa).
  - No existe un corte único para decir qué es "cercano"; 1.5 a 2.5 es una regla práctica muy usada.
- **Por qué importa:** si los errores están relacionados entre sí, el modelo parece más preciso de lo que realmente es. Es un problema típico de las series de tiempo (como en el notebook 21). Aquí cada auto es una venta independiente y las filas se mezclan al azar al dividir los datos, por lo que se espera un valor cercano a 2 y esta prueba es la menos relevante de las cuatro.

#### 4. Observaciones influyentes (distancia de Cook)

- **Qué es una observación influyente:** una observación que, **si se elimina, hace cambiar de forma notable los coeficientes del modelo**. Una observación influye mucho cuando combina dos cosas: (1) es muy distinta del resto en sus características (por ejemplo, un auto de lujo con un motor enorme) **y** (2) el modelo la predice mal. Si es muy distinta pero el modelo la predice bien, casi no influye; si se predice mal pero es un auto normal, tampoco influye mucho.
- **Distancia de Cook:** un número por observación que resume cuánto cambiarían los coeficientes si se eliminara esa observación. Reglas prácticas: **mayor a 1** = muy influyente (hay que revisarla); **mayor a 4/n** (n = número de observaciones) = vale la pena mirarla.
- **Qué hacer si hay influyentes:** no se eliminan automáticamente. Primero se revisa si el dato es un error (por ejemplo, un precio o un kilometraje mal digitado) y se compara el modelo con y sin esas observaciones: si los coeficientes casi no cambian, no es un problema.

In [ ]:
# Diagnóstico errores (residuos) de las predicciones de entrenamiento

#Se realiza la predicción sobre los datos de entrenamiento (en escala logarítmica)
prediccion_train = modelLRegresion.predict(X_train_const)

#Se calculan los residuos: diferencia entre la prediccion y el valor real en los datos de entrenamiento
residuos_train   = prediccion_train - y_train

# Pruebas sobre los supuestos
bp = het_breuschpagan(residuos_train, X_train_const)
jb = jarque_bera(residuos_train)
dw = durbin_watson(residuos_train)
cook = pd.Series(modelLRegresion.get_influence().cooks_distance[0], index=X_train_const.index)
umbral_cook = 4 / len(cook)

print("1. Homocedasticidad (Breusch-Pagan)")
print(f"   p-value = {bp[1]:.4f}")
if bp[1] < 0.05:
    print("   Se rechaza: la variabilidad de los errores NO es constante (el modelo se equivoca más en unos autos que en otros).")
    print("   Los coeficientes y las predicciones siguen siendo válidos, pero los p-values clásicos no son confiables: por eso se usan errores estándar robustos (HC3).")
else:
    print("   No se rechaza: no hay evidencia de que la variabilidad de los errores cambie; los errores estándar clásicos son confiables.")

print("\n2. Normalidad de los residuos (Jarque-Bera)")
print(f"   p-value = {jb[1]:.4f}")
if jb[1] < 0.05:
    print("   Se rechaza: los errores no tienen forma de campana (hay más errores extremos de lo normal o se inclinan hacia un lado).")
    print("   Con muchas observaciones los coeficientes y p-values siguen siendo aproximadamente válidos; revisa la gráfica Q-Q para ver qué tan grave es.")
else:
    print("   No se rechaza: los errores son compatibles con una distribución normal.")

print("\n3. Independencia de los errores (Durbin-Watson)")
print(f"   estadístico = {dw:.3f}")
if 1.5 <= dw <= 2.5:
    print("   Está entre 1.5 y 2.5 (cerca de 2): los errores son independientes entre sí.")
elif dw < 1.5:
    print("   Es menor a 1.5: el error de una observación se parece al de la anterior (autocorrelación positiva); los errores estándar podrían estar subestimados.")
else:
    print("   Es mayor a 2.5: el error cambia de signo de una observación a la siguiente (autocorrelación negativa).")

print("\n4. Observaciones influyentes (distancia de Cook)")
print(f"   Cook máxima = {cook.max():.3f}")
print(f"   Observaciones con Cook > 1 (muy influyentes): {(cook > 1).sum()}")
print(f"   Observaciones con Cook > 4/n = {umbral_cook:.4f} (vale la pena mirarlas): {(cook > umbral_cook).sum()} de {len(cook)}")
if (cook > 1).sum() == 0:
    print("   Ninguna observación es muy influyente por sí sola (Cook < 1): quitar un solo auto no cambia de forma importante el modelo.")
else:
    print("   Hay observaciones muy influyentes: revisa si sus datos son correctos y compara el modelo con y sin ellas.")

Estas son las 5 observaciones con mayor distancia de Cook, para ver cómo son los autos que más influyen. La segunda tabla compara los coeficientes con todas las observaciones y sin esas 5. Los coeficientes que casi no cambian son estables. Los que cambian mucho suelen ser los de variables con pocas filas (por ejemplo `Owner_Otro`) o muy relacionadas con otras (por ejemplo las medidas del vehículo): son menos estables y conviene interpretarlos con cautela.

In [ ]:
# Las 5 observaciones con mayor distancia de Cook
top_cook = cook.sort_values(ascending=False).head(5)

pd.DataFrame({'Cook': top_cook,
              'Precio real (USD)': np.exp(y_train[top_cook.index]),
              'Residuo (escala log)': residuos_train[top_cook.index]})

In [ ]:
# ¿Cambian los coeficientes si se quitan las 5 observaciones más influyentes?
modelo_sin_top = sm.OLS(y_train.drop(top_cook.index), X_train_const.drop(top_cook.index)).fit()

comparacion = pd.DataFrame({'con todas': modelLRegresion.params,
                            'sin las 5': modelo_sin_top.params})
comparacion['cambio (%)'] = 100 * (comparacion['sin las 5'] / comparacion['con todas'] - 1)

comparacion.round(4)

In [ ]:
# Gráficos para el pronóstico de residuos

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 10))

axes[0, 0].scatter(y_train, prediccion_train, edgecolors=(0, 0, 0), alpha = 0.4)
axes[0, 0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()],
                '--', color = 'black', lw=2)
axes[0, 0].set_title('Valor predicho vs valor real', fontsize = 10, fontweight = "bold")
axes[0, 0].set_xlabel('Real')
axes[0, 0].set_ylabel('Predicción')
axes[0, 0].tick_params(labelsize = 7)

axes[0, 1].scatter(list(range(len(y_train))), residuos_train,
                   edgecolors=(0, 0, 0), alpha = 0.4)
axes[0, 1].axhline(y = 0, linestyle = '--', color = 'black', lw=2)
axes[0, 1].set_title('Residuos del modelo', fontsize = 10, fontweight = "bold")
axes[0, 1].set_xlabel('id')
axes[0, 1].set_ylabel('Residuo')
axes[0, 1].tick_params(labelsize = 7)

sns.histplot(
    data    = residuos_train,
    stat    = "density",
    kde     = True,
    line_kws= {'linewidth': 1},
    color   =  "navy",#"firebrick",
    alpha   = 0.3,
    ax      = axes[1, 0]
)

axes[1, 0].set_title('Distribución residuos del modelo', fontsize = 10,
                     fontweight = "bold")
axes[1, 0].set_xlabel("Residuo")
axes[1, 0].tick_params(labelsize = 7)


sm.qqplot(
    residuos_train,
    fit   = True,
    line  = 'q',
    ax    = axes[1, 1],
    color = 'navy', #'firebrick',
    alpha = 0.4,
    lw    = 2
)
axes[1, 1].set_title('Q-Q residuos del modelo', fontsize = 10, fontweight = "bold")
axes[1, 1].tick_params(labelsize = 7)

axes[2, 0].scatter(prediccion_train, residuos_train,
                   edgecolors=(0, 0, 0), alpha = 0.4)
axes[2, 0].axhline(y = 0, linestyle = '--', color = 'black', lw=2)
axes[2, 0].set_title('Residuos del modelo vs predicción', fontsize = 10, fontweight = "bold")
axes[2, 0].set_xlabel('Predicción')
axes[2, 0].set_ylabel('Residuo')
axes[2, 0].tick_params(labelsize = 7)

# Se eliminan los axes vacíos
fig.delaxes(axes[2,1])

fig.tight_layout()
plt.subplots_adjust(top=0.9)
fig.suptitle('Diagnóstico residuos', fontsize = 12, fontweight = "bold");

## 7. Evaluación del modelo

El modelo se entrena con el **logaritmo** del precio, así que sus predicciones están en escala logarítmica. Para volver a dólares no basta con `np.exp(predicción)`: eso subestima el precio medio. Se multiplica por el promedio de `exp(error)` de entrenamiento (corrección de Duan, *smearing*). Las métricas en dólares son las que importan para el negocio.

### Entrenamiento

In [ ]:
# Factor de corrección para volver de logaritmo a dólares (smearing de Duan)
factor_smearing = np.mean(np.exp(-residuos_train))

# Error de Entrenamiento del modelo (escala logarítmica)
mse = mean_squared_error (y_true = y_train, y_pred = prediccion_train)

print(f"R2   de train (escala log): {r2_score(y_true = y_train, y_pred = prediccion_train):.4f}")
print(f"RMSE de train (escala log): {np.sqrt(mse):.4f}")
print(f"Factor de corrección (smearing): {factor_smearing:.4f}")

# Error de Entrenamiento del modelo (dólares)
metricas_train = metricas_precio(np.exp(y_train), np.exp(prediccion_train) * factor_smearing)
print("\nEntrenamiento (dólares):", {k: round(v, 4) for k, v in metricas_train.items()})

### Prueba interna (20 % de los datos)

In [ ]:
#Se extraen solo las columnas finales y se adiciona la constante
X_test_const = sm.add_constant(X_test[columnas_finales], has_constant='add')

prediccion_test = modelLRegresion.predict(X_test_const)

# Error de prueba (test) del modelo (escala logarítmica)
print(f"R2   de test (escala log): {r2_score(y_true = y_test, y_pred = prediccion_test):.4f}")
print(f"RMSE de test (escala log): {np.sqrt(mean_squared_error(y_true = y_test, y_pred = prediccion_test)):.4f}")

# Error de prueba (test) del modelo (dólares)
metricas_test = metricas_precio(np.exp(y_test), np.exp(prediccion_test) * factor_smearing)
print("\nPrueba interna (dólares):", {k: round(v, 4) for k, v in metricas_test.items()})

### Prueba real (`Prue`)

El conjunto `Prue` se apartó en el notebook 01 antes de cualquier ajuste y **no se usó en ninguna decisión del modelo** (ni en la selección de variables ni en el escalado), por lo que es la estimación más honesta del desempeño. No se le eliminan atípicos: es lo que el modelo encontraría en la práctica.

In [ ]:
#Carga del conjunto de prueba real
dp = pd.read_csv(local_dataset_path_prue)

#Precio real en dólares y logaritmo del precio (variable de salida del modelo)
precio_prue = dp['Price'].copy()
yp = np.log(dp['Price'])

#Dummies: sin 'drop_first' y luego se alinea con las columnas del entrenamiento
#(las categorías de referencia no existen en X; si falta alguna categoría en prueba, su dummy queda en 0)
Xp = pd.get_dummies(dp.drop(columns='Price'), dtype=int)
Xp = Xp.reindex(columns=X.columns, fill_value=0)

#Se escalan las variables numéricas con el escalador ajustado en entrenamiento
Xp[num_vars] = scaler.transform(Xp[num_vars])

#Predicción y retorno a dólares
Xp_const = sm.add_constant(Xp[columnas_finales], has_constant='add')
prediccion_prue = modelLRegresion.predict(Xp_const)
precio_pred_prue = np.exp(prediccion_prue) * factor_smearing

print(f"R2   de Prue (escala log): {r2_score(y_true = yp, y_pred = prediccion_prue):.4f}")
metricas_prue = metricas_precio(precio_prue, precio_pred_prue)
print("\nPrueba real (dólares):", {k: round(v, 4) for k, v in metricas_prue.items()})

In [ ]:
# Resumen de las métricas en dólares
pd.DataFrame({'Entrenamiento': metricas_train,
              'Prueba interna': metricas_test,
              'Prueba real (Prue)': metricas_prue}).T

### Análisis de los errores en dólares (prueba real)

In [ ]:
# Se arma una tabla con el precio real, el precio predicho y la diferencia absoluta
resultado = pd.DataFrame({'PriceReal': precio_prue.values, 'PricePredict': precio_pred_prue.values})
resultado['dif'] = (resultado['PriceReal'] - resultado['PricePredict']).abs()

# Las diferencias ordenadas
resultado.sort_values(by='dif', ascending=True)

In [ ]:
# Principales descripciones de las diferencias
print(resultado['dif'].describe())

In [ ]:
#Caja de bigotes de las diferencias de los precios

plt.figure(figsize=(6,4))

plt.title('Diferencia de precios de vehículos')
sns.boxplot(x=resultado['dif'], palette='nipy_spectral')

plt.show()

#### Referencias

- *Regresion Lineal en Python:*
    - https://www.cienciadedatos.net/documentos/py10-regresion-lineal-python.html
  
  
- *Machine learning con Python y Scikit-learn:*
    - https://www.cienciadedatos.net/documentos/py06_machine_learning_python_scikitlearn.html
    

- *scikit-learn:*    
    - https://scikit-learn.org/stable/
    - https://scikit-learn.org/stable/user_guide.html
  
  
- *Recursive Feature Elimination (RFE)*:
    - https://www.kaggle.com/goyalshalini93/car-price-prediction-linear-regression-rfe
    - https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html


- *Gráficas con  seaborn:*
    - https://ichi.pro/es/como-utilizar-python-seaborn-para-analisis-de-datos-exploratorios-28897898172180


- *Multicolinealidad:*
    - https://www.analyticslane.com/2020/04/22/solucionar-la-multicolinealidad-con-vif/


- *Analítica de grandes datos:*
    - https://jdvelasq.github.io/courses/analitica-de-grandes-datos/index.html